In [1]:
import sys
import altair as alt
import pandas as pd
from io import StringIO
import vl_convert as vlc

In [ ]:
data = pd.DataFrame({
    'Category': ['Physicochemical Properties', 'Sequence', 'Missed Cleavage'],
    'Percentage': [0.531, 0.344, 0.125],
    'Numbers': [17, 11, 4],
    'comb': ["(17) 53.1%", "(11) 34.4%", "(4) 12.5%"]
})
#Adding Pfly and DeepPD
#'Percentage': [0.533, 0.333, 0.133]
#'Percentage': [16, 10, 4]

midpoints = []
prev = 0
for i in data['Percentage'].to_list():
    midpoint = prev + (i/2)
    midpoints.append(midpoint)
    prev = prev + i
data["Midpoints"] = midpoints

# Define the color scheme with a palette of gray and black
color_scale_bar = alt.Scale(
    domain=['Physicochemical Properties', 'Sequence', 'Missed Cleavage'],
    range=['#AAAAAA', '#555555', '#000000']  # Black, dark gray, and light gray
)
color_scale_text = alt.Scale(
    domain=data['Midpoints'].to_list(),
    range=['#000000', '#FFFFFF', '#FFFFFF']  # Black, dark gray, and light gray
)
# Create the horizontal stacked bar chart
stacked_bar_chart = alt.Chart(data).mark_bar(size=50).encode(
    x=alt.X('Percentage:Q', stack='normalize', axis=None),
    color=alt.Color('Category:N', scale=color_scale_bar, legend=alt.Legend(orient='none', legendX=140, legendY=70, title=None, direction='horizontal')), #columnPadding=-30
    order=alt.Order('Percentage:Q', sort='descending')
)

white_text = alt.Chart(data).mark_text(size=12).encode(#color='white'
    x=alt.X('Midpoints:Q'),
    text=alt.Text('comb:N'),#, format=".1%"),#comment the format out when you go for total numbers
    color=alt.Color('Midpoints:Q', scale=color_scale_text, legend=None)
)

final_chart = (stacked_bar_chart+white_text).properties(width=600, height=50).configure_view(strokeOpacity=0)
final_chart.save("bar_chart_features_total_percent.pdf")
final_chart

alt.LayerChart(...)

In [4]:
import sys
data = pd.DataFrame({
    'Category': ['Traditional Machine Learning', 'Deep Learning'],
    'Percentage': [0.56, 0.44],
    'Numbers': [14, 11]
})

midpoints = []
prev = 0
for i in data['Percentage'].to_list():
    midpoint = prev + (i/2)
    midpoints.append(midpoint)
    prev = prev + i
data["Midpoints"] = midpoints

# Define the color scheme with a palette of gray and black
color_scale_bar = alt.Scale(
    domain=['Traditional Machine Learning', 'Deep Learning'],
    range=['#FFFFFF', '#000000']  # Black, dark gray, and light gray
)
color_scale_text = alt.Scale(
    domain=data['Midpoints'].to_list(),
    range=['#000000', '#FFFFFF']  # Black, dark gray, and light gray
)
# Create the horizontal stacked bar chart
stacked_bar_chart = alt.Chart(data).mark_bar(size=50, stroke="gray").encode(
    x=alt.X('Percentage:Q', stack='normalize', axis=None),
    color=alt.Color('Category:N', scale=color_scale_bar, legend=None),#, legend=alt.Legend(orient='bottom', title=None, columnPadding=30)),
    order=alt.Order('Percentage:Q', sort='descending')
)

white_text = alt.Chart(data).mark_text(size=12).encode(#color='white'
    x=alt.X('Midpoints:Q'),
    text=alt.Text('Percentage:Q', format=".1%"),#comment the format out when you go for total numbers
    color=alt.Color('Midpoints:Q', scale=color_scale_text, legend=None)
)

comb_chart = (stacked_bar_chart+white_text).properties(width=900, height=60)#.configure_view(strokeOpacity=1)
#comb_chart.save("Path/to/method_bar.svg")
comb_chart

alt.LayerChart(...)

In [ ]:
data_str = """Year\tTitle\tCategory
2006\tA computational approach toward label-free protein quantification using predicted peptide detectability:\tM
2007\tPrediction of peptides observable by mass spectrometry applied at the experimental set level\tM
2007\tPeptideSieve\tM
2008\tAPEX\tM
2008\tA support vector machine model for the prediction of proteotypic peptides for accurate mass and time proteomics\tM
2009\tESPPredictor\tM
2010\tThe Importance of Peptide Detectability for Protein Identification, Quantification, and Experiment Design in MS/MS Proteomics\tM
2011\tCONSeQuence\tM
2014\tPeptideRank\tM
2015\tPPA\tM
2015\tPREGO\tM
2017\tEnhanced Missing Proteins Detection in NCI60 Cell Lines Using an Integrative Search Engine Approach\tM
2018\td::pPop\tD
2019\tAP3\tM
2020\tDeepMSPeptide\tD
2020\tIn silico spectral libraries by deep learning facilitate data-Independant aquisition proteomics\tD
2021\tPepFormer\tD
2021\tCapsNet\tD
2022\tPD-BertEDL\tD
2023\tDeepDetect\tD
2023\tDbyDeep\tD
2023\tPeptideRanger\tM
2024\tKDEAN\tD
2024\tDeepPD\tD
2025\tPfly\tD"""

# Read data into DataFrame
data = pd.read_csv(StringIO(data_str), sep='\t')

# Count occurrences of 'M' and 'D' for each year
counts = data.groupby(['Year', 'Category']).size().unstack(fill_value=0).reset_index()

# Create cumulative sums
counts[['Traditional Machine Learning', 'Deep Learning']] = counts[['M', 'D']].cumsum()

# Build a DataFrame with explicit segment positions
rows = []
for _, row in counts.iterrows():
    year = row['Year']
    cum_tm = row['Traditional Machine Learning']
    cum_dl = row['Deep Learning']

    # Traditional Machine Learning segment (white, bottom)
    rows.append({
        'Year': year,
        'Category': 'Traditional Machine Learning',
        'Occurrences': cum_tm,      # this is the height of this segment
        'y0': 0,                    # bottom of segment
        'y1': cum_tm                # top of segment
    })

    # Deep Learning segment (black, top)
    rows.append({
        'Year': year,
        'Category': 'Deep Learning',
        'Occurrences': cum_dl,      # height of this segment
        'y0': cum_tm,               # starts where TM ends
        'y1': cum_tm + cum_dl       # top of stacked bar
    })

plot_df = pd.DataFrame(rows)
plot_df['y_center'] = (plot_df['y0'] + plot_df['y1']) / 2

# Bars: still using normal stacking
bars = alt.Chart(plot_df).mark_bar(stroke="gray").encode(
    x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('Occurrences:Q', title='Cumulative Occurrences', stack='zero'),
    color=alt.Color(
        'Category:N',
        scale=alt.Scale(
            domain=['Traditional Machine Learning', 'Deep Learning'],
            range=['white', 'black']
        ),
        legend=alt.Legend(orient="bottom", title=None, columnPadding=30)
    ),
    # Deep Learning on top of Traditional ML
    order=alt.Order('Category:N', sort='descending')
).properties(
    width=450,
    height=250,
)

# Labels: positioned at the precomputed y_center (no stacking here)
labels = alt.Chart(plot_df).transform_filter(
    'datum.Occurrences > 0'  # avoid labels for zero-height segments
).mark_text(fontSize=11).encode(
    x=alt.X('Year:O'),
    y=alt.Y('y_center:Q'),
    text=alt.Text('Occurrences:Q'),
    color=alt.condition(
        alt.datum.Category == 'Deep Learning',
        alt.value('white'),  # white text on black bars
        alt.value('black')   # black text on white bars
    )
)

stacked_bar_chart_new = bars + labels

stacked_bar_chart_new
stacked_bar_chart_new.save("M_D_occurences.pdf")


In [ ]:
import plotly.graph_objects as go
import pandas as pd

data = pd.DataFrame({
    'Theme': ['Machine Learning 100%'] * 13,
    'Type': [
        'Traditional Machine Learning 56%', 'Traditional Machine Learning 56%', 'Traditional Machine Learning 56%',
        'Traditional Machine Learning 56%', 'Traditional Machine Learning 56%',
        'Deep Learning 44%', 'Deep Learning 44%', 'Deep Learning 44%', 'Deep Learning 44%',
        'Deep Learning 44%', 'Deep Learning 44%', 'Deep Learning 44%', 'Deep Learning 44%'
    ],
    'Category': [
        'Random Forest<br>Classifier 20%',
        'Multilayer<br>Perceptron 20%',
        'Ensemble of<br>different<br>Methods 8%',
        'Support<br>Vector<br>Machines 4%',
        'Gaussian<br>Mixture<br>Likelihood<br>Function 4%',
        'Bidirectional<br>Long<br>Short-Term<br>Memory<br>Network 12%',
        'Convolutional<br>Neural<br>Network 8%',
        'Feed<br>Forward<br>Neural<br>Network 4%',
        'Long<br>Short-Term<br>Memory<br>Network 4%',
        'Hybrid<br>Neural<br>Network 4%',
        'Transformer 4%',
        'Bidirectional<br>Recurrent<br>Neural<br>Network and<br>Gated<br>Recurrent<br>Units 4%',
        'Transformer and<br>Gated<br>Recurrent<br>Units 4%'
    ],
    'Percentage': [
        20, 20, 8, 4, 4,
        12, 8, 4, 4, 4, 4, 4, 4
    ]
})

# --- compute higher-level percentages explicitly ---
type_pct = data.groupby('Type')['Percentage'].sum().to_dict()       # e.g. 60.87, 39.13
theme_pct = data.groupby('Theme')['Percentage'].sum().to_dict()     # e.g. 100

# Build node lists (leaf categories + types + theme)
labels = list(data['Category']) + list(type_pct.keys()) + list(theme_pct.keys())
parents = (
    list(data['Type']) +               # each category’s parent is its Type
    ['Machine Learning 100%'] * len(type_pct) +  # each Type’s parent is Theme
    ['']                               # root has no parent
)

# Areas = percentages for all nodes (sums at higher levels)
values = (
    list(data['Percentage']) +
    list(type_pct.values()) +
    list(theme_pct.values())
)

# Colors = we want them to match these same percentages (0–100 scale)
colors = values

fig = go.Figure(
    go.Treemap(
        labels=labels,
        parents=parents,
        values=values,
        branchvalues='total',
        marker=dict(
            colors=colors,
            colorscale=[[0, '#eff3ff'],
                        [0.25, '#bdd7e7'],
                        [0.5, '#6baed6'],
                        [0.75, '#3182bd'],
                        [1, '#08519c']],
            cmin=0,
            cmax=100,
            colorbar=dict(
                title='Percentage',
                orientation='h',
                x=0.5,
                xanchor='center',
                y=-0.4
            )
        ),
        textinfo='label'
    )
)

fig.update_traces(textfont_size=14)
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()

# To save:
fig.write_image(file="treemap_techniques.pdf", scale=3)